In [ ]:
# ======================================================================
# STEP W1 (UPDATED FOR ERA5 CSV)
# WEATHER DATA PREPROCESSING + QUALITY AUDIT
#
# INPUT:
#   bangladesh_weather_era5_2015_2026.csv
#
# OUTPUT:
#   bangladesh_national_hourly_weather.csv
# ======================================================================


import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import drive
from IPython.display import display


warnings.filterwarnings("ignore")

pd.set_option(
    "display.max_columns",
    200
)

pd.set_option(
    "display.width",
    250
)


print("="*110)
print("STEP W1: ERA5 WEATHER PREPROCESSING + QUALITY AUDIT")
print("="*110)


# ======================================================================
# 1. DRIVE
# ======================================================================

drive.mount(
    "/content/drive"
)


# ======================================================================
# 2. PATHS
# ======================================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Load_Forecasting_Paper"
)


WEATHER_RAW_DIR = (
    PROJECT_ROOT
    /
    "data"
    /
    "weather"
    /
    "raw"
)


WEATHER_PROCESSED_DIR = (
    PROJECT_ROOT
    /
    "data"
    /
    "weather"
    /
    "processed"
)


RESULT_DIR = (
    PROJECT_ROOT
    /
    "results"
    /
    "weather_extension"
)


FIGURE_DIR = (
    PROJECT_ROOT
    /
    "figures"
    /
    "weather_extension"
)



for directory in [

    WEATHER_PROCESSED_DIR,
    RESULT_DIR,
    FIGURE_DIR

]:

    directory.mkdir(
        parents=True,
        exist_ok=True
    )



# ======================================================================
# 3. ERA5 INPUT FILE
# ======================================================================


WEATHER_FILE = (

    WEATHER_RAW_DIR
    /
    "bangladesh_weather_era5_2015_2026.csv"

)



if not WEATHER_FILE.exists():

    raise FileNotFoundError(

        f"""
ERA5 weather file not found:

{WEATHER_FILE}
"""

    )



print(
    "\nUsing weather file:"
)

print(
    WEATHER_FILE
)



# ======================================================================
# 4. LOAD ERA5 WEATHER
# ======================================================================


weather = pd.read_csv(

    WEATHER_FILE,

    low_memory=False

)



print("\nRaw weather:")

print(
    "Rows:",
    f"{len(weather):,}"
)

print(
    "Columns:",
    len(weather.columns)
)


display(
    weather.head()
)



# ======================================================================
# 5. COLUMN CHECK
# ======================================================================


required_columns = [

    "time",

    "temperature_2m",

    "relative_humidity_2m",

    "rain",

    "pressure_msl",

    "cloud_cover",

    "wind_speed_100m",

    "wind_direction_100m",

    "soil_temperature_100_to_255cm",

    "city"

]


missing = [

    c

    for c in required_columns

    if c not in weather.columns

]


if missing:

    raise ValueError(

        f"Missing columns: {missing}"

    )


print(
    "\nRequired columns found."
)



# ======================================================================
# 6. TIME PROCESSING
# ======================================================================


weather["datetime"] = pd.to_datetime(

    weather["time"],

    errors="coerce"

)



weather = (

    weather

    .dropna(
        subset=[
            "datetime"
        ]
    )

    .sort_values(
        [
            "datetime",
            "city"
        ]
    )

    .reset_index(
        drop=True
    )

)



print("\nCoverage:")

print(
    "Start:",
    weather["datetime"].min()
)


print(
    "End:",
    weather["datetime"].max()
)


print(
    "Cities:",
    weather["city"].nunique()
)



# ======================================================================
# 7. DUPLICATE CHECK
# ======================================================================


duplicate_count = (

    weather

    .duplicated(

        subset=[
            "datetime",
            "city"
        ]

    )

    .sum()

)



print(

    "\nDuplicate city-hour rows:",

    duplicate_count

)



if duplicate_count > 0:

    weather = (

        weather

        .drop_duplicates(

            subset=[
                "datetime",
                "city"
            ],

            keep="first"

        )

    )



# ======================================================================
# 8. NUMERIC CONVERSION
# ======================================================================


numeric_columns = [

    c

    for c in required_columns

    if c not in [
        "time",
        "city"
    ]

]


for col in numeric_columns:

    weather[col] = pd.to_numeric(

        weather[col],

        errors="coerce"

    )



# ======================================================================
# 9. MISSING AUDIT
# ======================================================================


missing_report = pd.DataFrame({

    "missing_count":
        weather[numeric_columns]
        .isna()
        .sum(),

    "missing_percent":
        weather[numeric_columns]
        .isna()
        .mean()
        *
        100

})


print("\nMissing values:")

display(
    missing_report
)



# ======================================================================
# 10. DERIVED FEATURES
# ======================================================================


weather["temp_humidity_interaction"] = (

    weather["temperature_2m"]

    *

    weather["relative_humidity_2m"]

)



weather["cooling_degree_24"] = (

    weather["temperature_2m"]

    -

    24

).clip(
    lower=0
)



weather["rain_event"] = (

    weather["rain"]

    >

    0

).astype(int)



# ======================================================================
# 11. NATIONAL AGGREGATION
# ======================================================================


national = (

    weather

    .groupby(
        "datetime"
    )

    .agg(

        weather_city_count=
        (
            "city",
            "nunique"
        ),


        temp_mean=
        (
            "temperature_2m",
            "mean"
        ),

        temp_min=
        (
            "temperature_2m",
            "min"
        ),

        temp_max=
        (
            "temperature_2m",
            "max"
        ),


        humidity_mean=
        (
            "relative_humidity_2m",
            "mean"
        ),


        humidity_max=
        (
            "relative_humidity_2m",
            "max"
        ),


        rain_mean=
        (
            "rain",
            "mean"
        ),


        rain_max=
        (
            "rain",
            "max"
        ),


        rain_city_fraction=
        (
            "rain_event",
            "mean"
        ),


        pressure_mean=
        (
            "pressure_msl",
            "mean"
        ),


        cloud_mean=
        (
            "cloud_cover",
            "mean"
        ),


        wind_mean=
        (
            "wind_speed_100m",
            "mean"
        ),


        wind_max=
        (
            "wind_speed_100m",
            "max"
        ),


        wind_direction_mean=
        (
            "wind_direction_100m",
            "mean"
        ),


        soil_temperature_mean=
        (
            "soil_temperature_100_to_255cm",
            "mean"
        ),


        cooling_degree_mean=
        (
            "cooling_degree_24",
            "mean"
        ),


        temp_humidity_mean=
        (
            "temp_humidity_interaction",
            "mean"
        )

    )

    .reset_index()

)



national["temp_range_spatial"] = (

    national["temp_max"]

    -

    national["temp_min"]

)



# ======================================================================
# 12. COVERAGE CHECK
# ======================================================================


full_range = pd.date_range(

    national["datetime"].min(),

    national["datetime"].max(),

    freq="h"

)


missing_hours = (

    full_range

    .difference(

        pd.DatetimeIndex(

            national["datetime"]

        )

    )

)



print("\nNational weather:")

print(
    "Rows:",
    f"{len(national):,}"
)


print(
    "Missing hourly timestamps:",
    len(missing_hours)
)



# ======================================================================
# 13. SAVE
# ======================================================================


OUTPUT_FILE = (

    WEATHER_PROCESSED_DIR

    /

    "bangladesh_national_hourly_weather.csv"

)



national.to_csv(

    OUTPUT_FILE,

    index=False

)



print("\nSaved:")

print(
    OUTPUT_FILE
)



# ======================================================================
# 14. FORECAST COVERAGE CHECK
# ======================================================================


FORECAST_DIR = (

    PROJECT_ROOT

    /

    "data"

    /

    "forecasting"

)



for name in [

    "train_2015_2022.csv",

    "validation_2023.csv",

    "test_2024_2026.csv"

]:

    path = FORECAST_DIR / name

    df = pd.read_csv(
        path
    )

    df["target_date"] = pd.to_datetime(
        df["target_date"]
    )


    coverage = (

        df["target_date"]

        -

        pd.Timedelta(days=1)

    ).between(

        national["datetime"].min(),

        national["datetime"].max()

    )


    print(
        name,
        ":",
        f"{coverage.mean()*100:.2f}%"
    )



print("\n")
print("="*110)
print("STEP W1 COMPLETED")
print("="*110)

print(
    "Next: Run W2 unchanged."
)

STEP W1: ERA5 WEATHER PREPROCESSING + QUALITY AUDIT
Mounted at /content/drive

Using weather file:
/content/drive/MyDrive/Load_Forecasting_Paper/data/weather/raw/bangladesh_weather_era5_2015_2026.csv

Raw weather:
Rows: 383,520
Columns: 10


,time,temperature_2m,relative_humidity_2m,rain,pressure_msl,cloud_cover,wind_speed_100m,wind_direction_100m,soil_temperature_100_to_255cm,city
0,2015-04-01T00:00,23.2,92,0.0,1010.8,45,16.1,151,25.1,Chittagong
1,2015-04-01T01:00,23.3,90,0.0,1010.4,59,14.7,158,25.1,Chittagong
2,2015-04-01T02:00,23.3,90,0.0,1010.0,55,13.5,164,25.1,Chittagong
3,2015-04-01T03:00,23.2,89,0.0,1009.8,69,12.7,165,25.1,Chittagong
4,2015-04-01T04:00,23.7,89,0.0,1010.4,73,12.9,162,25.1,Chittagong



Required columns found.

Coverage:
Start: 2015-04-01 00:00:00
End: 2026-03-08 23:00:00
Cities: 4

Duplicate city-hour rows: 0

Missing values:


,missing_count,missing_percent
temperature_2m,0,0.0
relative_humidity_2m,0,0.0
rain,0,0.0
pressure_msl,0,0.0
cloud_cover,0,0.0
wind_speed_100m,0,0.0
wind_direction_100m,0,0.0
soil_temperature_100_to_255cm,0,0.0



National weather:
Rows: 95,880
Missing hourly timestamps: 0

Saved:
/content/drive/MyDrive/Load_Forecasting_Paper/data/weather/processed/bangladesh_national_hourly_weather.csv
train_2015_2022.csv : 100.00%
validation_2023.csv : 100.00%
test_2024_2026.csv : 100.00%


STEP W1 COMPLETED
Next: Run W2 unchanged.
